## Create Interactive Plots with Plotly
In this notebook, we will learn how to use `plotly` library to build interactive plots.

Plotly library: Plotly's Python graphing library makes interactive, publication-quality graphs online. Examples of how to make line plots, scatter plots, area charts, bar charts, error bars, box plots, histograms, heatmaps, subplots, multiple-axes, polar charts, and bubble charts.

In [1]:
# Run this if plotly is not installed yet
#!pip install plotly==5.24.1
# !pip install plotly.express --quiet
# !pip show plotly

In [2]:
### Importing libraries
import numpy as np
import pandas as pd

# plotly libraries
from plotly.offline import init_notebook_mode, iplot, plot
import plotly as py
init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.express as px

# matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

Matplotlib is building the font cache; this may take a moment.


In [3]:
# Retain the helper used throughout the notebook with a current renderer.
def configure_plotly_browser_state():
    import sys
    import plotly.io as pio

    pio.renderers.default = (
        "colab" if "google.colab" in sys.modules else "notebook_connected"
    )


## University World rank dataset
University ranking dataset based on measurements captured by Times Higher Education World University Ranking.

Dataset fields:
- `world_rank` -  world rank for the university
- `university_name` -  name of university
- `country` - country of each university
- `teaching` - university score for teaching (the learning environment)
- `international` - university score international outlook (staff, students, research)
- `research` - university score for research (volume, income and reputation)
- `citations` - university score for citations (research influence)
- `income` - university score for industry income (knowledge transfer)
- `total_score` - total score for university, used to determine rank
- `num_students` - number of students at the university
- `student_staff_ratio` - Number of students divided by number of staff
- `international_students` - Percentage of students who are international
- `female_male_ratio` - Female student to Male student ratio
- `year` - year of the ranking (2011 to 2016 included)

In [4]:
# University World rank dataset
dataset = "https://raw.githubusercontent.com/csbfx/advpy122-data/master/timesData.csv"
uni = pd.read_csv(dataset)

In [5]:
# uni.info()
# uni.head()

In [6]:
### Notice the "world_rank" column and the extra string.
## These strings in the column that need to be remove
uni[uni.world_rank.str.contains('=')]

,world_rank,university_name,country,teaching,international,research,citations,income,total_score,num_students,student_staff_ratio,international_students,female_male_ratio,year
1841,=39,"University of California, San Diego",United States of America,56.9,42.9,69.8,98.7,56.7,72.2,"27,233",6.5,11%,48 : 52,2016
1842,=39,"University of California, Santa Barbara",United States of America,52.6,61.5,66.0,99.2,90.4,72.2,"22,020",27.3,11%,52 : 48,2016
1846,=44,"University of California, Davis",United States of America,60.1,58.4,72.7,84.3,57.3,71.0,"35,364",13.9,13%,54 : 46,2016
1847,=44,University of Hong Kong,Hong Kong,64.6,99.5,72.8,70.1,53.7,71.0,"19,835",17.6,38%,53 : 47,2016
1849,=47,Tsinghua University,China,73.3,39.5,83.0,58.8,100.0,70.0,"39,763",13.7,10%,32 : 68,2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1993,=190,Boston College,United States of America,34.1,60.0,29.3,83.3,46.8,49.6,"13,216",17.4,19%,54 : 46,2016
1995,=193,University of Luxembourg,Luxembourg,25.0,99.8,26.7,84.8,38.1,49.4,"5,144",15.9,52%,50 : 50,2016
1996,=193,Texas A&M University,United States of America,49.4,47.8,52.4,47.1,46.4,49.4,"50,657",21.4,9%,47 : 53,2016
1998,=196,Newcastle University,United Kingdom,30.9,84.3,27.5,81.5,34.7,49.2,"20,174",15.2,29%,50 : 50,2016


### `str.replace()`
How do we remove the "=" string in the "world_rank column"?   
We can replace a string with the method `str.replace(<org>, <new>)`. After we have remove the string, you can use `pd.to_numberic()` to convert the column to numeric and the parameter `errors='coerce'` to handle columns with the value NaN.

In [7]:
# Separate tied ranks (for example, "=25") from the other rows.
tied_rank = uni["world_rank"].astype("string").str.contains("=", na=False)
duni = uni.loc[~tied_rank].copy()
dunib = uni.loc[tied_rank].copy()
dunib["world_rank"] = dunib["world_rank"].str.replace("=", "", regex=False)

# Rank bands such as "201-250" are not individual numeric ranks.
duni["world_rank"] = pd.to_numeric(duni["world_rank"], errors="coerce")
dunib["world_rank"] = pd.to_numeric(dunib["world_rank"], errors="coerce")

# Restore source row order so the later iloc examples remain ranked.
uni = pd.concat([duni, dunib], axis=0, sort=False).sort_index()
uni[["university_name", "world_rank", "year"]].head()


,university_name,world_rank,year
0,Harvard University,1.0,2011
1,California Institute of Technology,2.0,2011
2,Massachusetts Institute of Technology,3.0,2011
3,Stanford University,4.0,2011
4,Princeton University,5.0,2011


In [8]:
# uni.info()

### RBG Colors
You can provide rbg colors for most plots. An easy way is to select a color using Google's color picker (Google "Color Picker") or you can randomly generate a rbg color below. I also like [Color Brewer](https://colorbrewer2.org/#type=sequential&scheme=BuGn&n=3) for selecting colors that are color blind friendly.

In [9]:
# Random RBG color generator
import numpy as np
color = list(np.random.choice(range(256), size=3))
print(color)

[255, 214, 207]


## Line Chart
Let's plot `citation` and `teaching` vs `world_rank` of Top 100 Universities

We will use plotly `Scatter` object. Ref: [plotly.graph_objects.Scatter](https://plotly.com/python-api-reference/generated/plotly.graph_objects.Scatter.html). In addition, we will create traces which are graph objects.

A trace object will contain the actual data (such as the x and y coordinates),  along with all the associated visual attributes like color, size, line style, and marker shape. Using traces, we can interact with the plot by clicking on a legend item to hide or show that trace.

In [10]:
### Run configure_plotly_browser_state() if you are in Google colab
configure_plotly_browser_state()

### Get the top 100 universities
df = uni.iloc[:100,:]

### import graph objects as "go"
import plotly.graph_objs as go

### Creating trace1
trace1 = go.Scatter(
                    x = df.world_rank,
                    y = df.citations,
                    mode = "lines",
                    name = "citations",
                    marker = dict(color = 'rgba(172, 135, 11, 0.8)'),
                    text= df.university_name)

### Creating trace2
trace2 = go.Scatter(
                    x = df.world_rank,
                    y = df.teaching,
                    mode = "lines+markers",
                    name = "teaching",
                    marker = dict(color = 'rgba(127, 207, 159, 0.8)'),
                    text= df.university_name)

data = [trace1, trace2]

layout = dict(title = 'Citation and Teaching vs World Rank of Top 100 Universities',
              xaxis= dict(title= 'World Rank', ticklen= 5, zeroline= False)
             )

fig = dict(data = data, layout = layout)
iplot(fig)

### Plotly express, `px`
Plotly express, `px`, is a higher-level API that allows us to quickly create interactive plots. It requires minimal code compare to Plotly Graph Objects (`go`), which is more verbose but enables us to have more customization.

In [11]:
# Plotly express way
configure_plotly_browser_state()
line_plot = px.line(df,
                    x="world_rank",
                    y="citations",
                    hover_data="university_name",
                    title="World ranking relationship to number of citations")

trace2 = go.Scatter(
                    x = df.world_rank,
                    y = df.teaching,
                    mode = "lines+markers",
                    name = "teaching",
                    marker = dict(color = 'rgba(127, 207, 159, 0.8)'),
                    text= df.university_name)

line_plot.add_trace(trace2)
line_plot.show()

In [12]:
### Creating a long-form table of teaching and citation for top 100
df = uni.iloc[:100,:]
for_color_test = df.melt(id_vars=[col for col in df.columns if col not in ["teaching", "citations"]], var_name="Score Category", value_name="score")
for_color_test

### Plotly express to create interactive plot
configure_plotly_browser_state()
line_plot = px.line(for_color_test,
                    x="world_rank",
                    y="score",
                    hover_data="university_name",
                    color="Score Category",
                    title="Melt is so useful sometimes",
                    line_dash = "Score Category")

line_plot.show()

### CL14.1 Create a line plot
Create a plot for the Top 25 Universities showing the citation, research and teaching measurements (score). Allow hover to show university's names.

Select the 25 best-ranked universities in 2016. Create a line plot showing the citation, research and teaching measurements (score). Allow hover to show university's names.

<!-- top25_2016 = practice.loc[practice["year"].eq(2016)].dropna(subset=["rank_clean"]).nsmallest(25,"rank_clean")
print(top25_2016.shape)
print(top25_2016[["citations","research","teaching"]].isna().sum())
display(top25_2016[["university_name","rank_clean","citations","research","teaching"]]) -->

In [13]:
configure_plotly_browser_state()

top25_2016 = (
    uni.loc[uni["year"].eq(2016)]
    .dropna(subset=["world_rank"])
    .nsmallest(25, "world_rank")
    .copy()
)
top25_scores = top25_2016.melt(
    id_vars=["university_name", "world_rank"],
    value_vars=["citations", "research", "teaching"],
    var_name="Score category",
    value_name="Score",
)
top25_line = px.line(
    top25_scores,
    x="world_rank",
    y="Score",
    color="Score category",
    markers=True,
    hover_name="university_name",
    labels={"world_rank": "World rank"},
    title="Citations, Research and Teaching: Top 25 Universities in 2016",
)
top25_line.show()


# Scatter plot with plotly `go.Scatter()`
Create a scatter plot for "citations" vs "world_rank" of top 100 universities for 2012, 2014 and 2016 using plotly go object.

In [14]:
# Run configure_plotly_browser_state() if you are in Google colab
configure_plotly_browser_state()

df2012 = uni[uni.year == 2012].iloc[:100,:]
df2014 = uni[uni.year == 2014].iloc[:100,:]
df2016 = uni[uni.year == 2016].iloc[:100,:]

## Create trace1 (2012)
trace1 =go.Scatter(
                    x = df2012.world_rank,
                    y = df2012.citations,
                    mode = "markers",
                    name = "2012",
                    marker = dict(color = 'rgba(31, 237, 108, 0.8)'),
                    text= df2012.university_name)

## Create trace2 (2014)
trace2 =go.Scatter(
                    x = df2014.world_rank,
                    y = df2014.citations,
                    mode = "markers",
                    name = "2014",
                    marker = dict(color = 'rgba(245, 73, 197, 0.8)'),
                    text= df2014.university_name)

## Create trace3 (2016)
trace3 =go.Scatter(
                    x = df2016.world_rank,
                    y = df2016.citations,
                    mode = "markers",
                    name = "2016",
                    marker = dict(color = 'rgba(94, 29, 211, 0.8)'),
                    text= df2016.university_name)

data = [trace1, trace2, trace3]
layout = dict(title = 'Citation vs world rank of top 100 universities for 2012, 2014 and 2016',
              xaxis= dict(title= 'World Rank',ticklen=10,zeroline= False),
              yaxis= dict(title= 'Citation',ticklen= 5,zeroline= False)
             )

fig = dict(data = data, layout = layout)
iplot(fig)

In [15]:
## Same plot but with plotly express
configure_plotly_browser_state()
px.scatter(uni,
           x="world_rank",
           y="citations",
           color="year",
           title="Citation vs world rank of top 100 universities for 2012, 2014 and 2016"
           )

### CL14.2 Create a hover-rich scatter plot

Plot citations against numeric world rank for the top 100 universities in 2016. Add hover labels for university and teaching score. Describe any association you observe (this does not necessary infer causation).

Why might a scatter plot be preferable over a line plot for investigating universities and ranking?



In [16]:
configure_plotly_browser_state()

top100_2016 = (
    uni.loc[uni["year"].eq(2016)]
    .dropna(subset=["world_rank"])
    .nsmallest(100, "world_rank")
    .copy()
)
citation_scatter = px.scatter(
    top100_2016,
    x="world_rank",
    y="citations",
    hover_name="university_name",
    hover_data={"teaching": ":.1f", "world_rank": True, "citations": ":.1f"},
    labels={
        "world_rank": "World rank",
        "citations": "Citation score",
        "teaching": "Teaching score",
    },
    title="Citation Score vs World Rank: Top 100 Universities in 2016",
)
citation_scatter.show()


In [17]:
rank_citation_correlation = top100_2016["world_rank"].corr(top100_2016["citations"])
print(f"Pearson correlation between numeric rank and citation score: {rank_citation_correlation:.3f}")


Pearson correlation between numeric rank and citation score: -0.501


**CL14.2 explanation:** Citation scores tend to decrease as numeric
world rank increases (a larger rank number means a lower ranking),
although there is substantial variation. The Pearson correlation for
these 100 universities is approximately **-0.501**, indicating a
moderate negative association. This does not establish causation;
citation scores also contribute to the ranking itself.

A scatter plot is preferable because universities are separate
observations. Connecting them with lines suggests continuity or
meaningful intermediate values between institutions. Individual points
make the spread, outliers, and multiple universities tied at a rank
easier to examine.


## Bar Charts using `go.Bar()`
Let's plot "citations" and "teaching" of top 10 universities in 2016

In [18]:
# Run configure_plotly_browser_state() if you are in Google colab
configure_plotly_browser_state()

df2016 = uni[uni.year == 2016].iloc[:10,:]


## Create trace1 (2016 citations)
trace1 = go.Bar(
                x = df2016.university_name,
                y = df2016.citations,
                name = "citations",
                marker = dict(color = 'rgba(123, 153, 23, 0.5)',
                             line=dict(color='rgb(0,0,0)',width=1.5)),
                text = df2016.country)

## Create trace2 (2016 teaching)
trace2 = go.Bar(
                x = df2016.university_name,
                y = df2016.teaching,
                name = "teaching",
                marker = dict(color = 'rgba(94, 29, 211, 0.5)',
                              line=dict(color='rgb(0,0,0)',width=1.5)),
                text = df2016.country)

data = [trace1, trace2]
layout = go.Layout(title = 'Citation and teaching for top 10 universities in 2016',
                   barmode = "group",
                   margin=dict(r=200, l=110, b=150, t=60))

fig = go.Figure(data = data, layout = layout)
iplot(fig)

### Stack bar plots using `'type': 'bar'`
A stacked bar plot is a way to extend a standard bar plot by dividing each bar into smaller segments to show how a total amount breaks down into smaller sub-components. With a stacked bar plot we can compare the total values for different components that makes up each bar.

In [19]:
### Run configure_plotly_browser_state() if you are in Google colab
configure_plotly_browser_state()

### Stack plots
x = df2016.university_name

trace1 = {
  'x': x,
  'y': df2016.citations,
  'name': 'citation',
  'type': 'bar'
};

trace2 = {
  'x': x,
  'y': df2016.teaching,
  'name': 'teaching',
  'type': 'bar'
};

data = [trace1, trace2];
layout = {
  'barmode': 'relative', # makes it a stacked bar plot
  'title': 'citations and teaching of top 10 universities in 2016',
  'margin': dict(r=200, l=110, b=180, t=70) # margins useful when you need extra space for things like axis labels, legends, or titles
};

fig = go.Figure(data = data, layout = layout)
iplot(fig)

## CL14.4: Plot bar plot and stacked bar plot
Create two plots; first is a bar plot showing the top 50 universities measurements for citations, research and teaching.

Convert that plot into a stacked plot. Why might it be easier to compare within universities than a stacked bar?

In [20]:
configure_plotly_browser_state()

# Continue using the 2016 rankings from the preceding exercises.
top50_2016 = (
    uni.loc[uni["year"].eq(2016)]
    .dropna(subset=["world_rank"])
    .nsmallest(50, "world_rank")
    .copy()
)
top50_scores = top50_2016.melt(
    id_vars=["university_name", "world_rank"],
    value_vars=["citations", "research", "teaching"],
    var_name="Score category",
    value_name="Score",
)
university_order = top50_2016["university_name"].tolist()
grouped_bar = px.bar(
    top50_scores,
    x="university_name",
    y="Score",
    color="Score category",
    barmode="group",
    hover_name="university_name",
    hover_data={"world_rank": True},
    category_orders={"university_name": university_order},
    labels={"university_name": "University", "world_rank": "World rank"},
    title="Citations, Research and Teaching: Top 50 Universities in 2016",
    width=1800,
    height=850,
)
grouped_bar.update_xaxes(tickangle=-60, automargin=True)
grouped_bar.update_layout(margin=dict(b=330))
grouped_bar.show()


In [21]:
configure_plotly_browser_state()

stacked_bar = go.Figure(grouped_bar)
stacked_bar.update_layout(
    barmode="stack",
    title="Stacked Citation, Research and Teaching Scores: Top 50 Universities in 2016",
    yaxis_title="Sum of component scores (not the overall ranking score)",
)
stacked_bar.show()


**CL14.4 explanation:** Grouped bars make it easier to compare citation,
research, and teaching scores within each university because all three
start at the same zero baseline. In a stacked chart, the upper segments
start at different heights, making their lengths harder to compare.
The stacked version emphasizes the sum and relative contributions,
but these separate scores are not parts of a natural whole. Their
unweighted sum is not the university's official overall ranking score.


## Pie Chart
Let's make a pie chart for `Students rate` of top 5 universities in 2016.

In [22]:
df2016 = uni[uni.year == 2016].iloc[:5,:]
df2016

,world_rank,university_name,country,teaching,international,research,citations,income,total_score,num_students,student_staff_ratio,international_students,female_male_ratio,year
1803,1.0,California Institute of Technology,United States of America,95.6,64.0,97.6,99.8,97.8,95.2,"2,243",6.9,27%,33 : 67,2016
1804,2.0,University of Oxford,United Kingdom,86.5,94.4,98.9,98.8,73.1,94.2,"19,919",11.6,34%,46 : 54,2016
1805,3.0,Stanford University,United States of America,92.5,76.3,96.2,99.9,63.3,93.9,"15,596",7.8,22%,42 : 58,2016
1806,4.0,University of Cambridge,United Kingdom,88.2,91.5,96.7,97.0,55.0,92.8,"18,812",11.8,34%,46 : 54,2016
1807,5.0,Massachusetts Institute of Technology,United States of America,89.4,84.0,88.6,99.7,95.4,92.0,"11,074",9.0,33%,37 : 63,2016


In [23]:
# Run configure_plotly_browser_state() if you are in Google colab
configure_plotly_browser_state()

df2016 = uni[uni.year == 2016].iloc[:5,:]
pie1 = df2016.num_students
pie1_list = [float(each.replace(',', '')) for each in df2016.num_students]  # str(2,4) => str(2.4) = > float(2.4) = 2.4
labels = df2016.university_name
# figure
fig = {
  "data": [
    {
      "values": pie1_list,
      "labels": labels,
      "domain": {"x": [0, .5]},
      "name": "Student info",
      "text": pie1_list,
       "hovertemplate":"%{label}: <br>Percent: %{percent} </br> # of Students: %{text}",
      "hole": .3, # adding a hole in the middle
      "type": "pie"
    },],
  "layout": {
        "title":"Number of students from the top 5 universities",
        "annotations": [
            { "font": { "size": 12},
              "showarrow": False,
              "text": " ",
                "x": 0.5,
                "y": 0.8
            },
        ]
    }
}
iplot(fig)

## Bubble plots
Let's create a bubble plot for the `University world rank` (first 20) vs `teaching score` with `number of students`(size) and `international score`(color) in 2016

In [24]:
configure_plotly_browser_state()

df2016 = uni[uni.year == 2016].iloc[:20, :]
num_students_size = pd.to_numeric(
    df2016.num_students.str.replace(",", "", regex=False), errors="coerce"
)
international_color = pd.to_numeric(df2016.international, errors="coerce")

fig = go.Figure(go.Scatter(
    x=df2016.world_rank,
    y=df2016.teaching,
    mode="markers",
    text=df2016.university_name,
    marker=dict(
        color=international_color,
        size=num_students_size,
        sizemode="area",
        sizeref=2 * num_students_size.max() / 60**2,
        sizemin=4,
        showscale=True,
        colorbar=dict(title="International score"),
    ),
    customdata=num_students_size,
    hovertemplate=(
        "%{text}<br>World rank: %{x}<br>Teaching score: %{y}"
        "<br>Students: %{customdata:,.0f}"
        "<br>International score: %{marker.color}<extra></extra>"
    ),
))
fig.update_layout(
    title="Top 20 Universities in 2016: Students (Size), International Score (Color)",
    xaxis_title="University world rank",
    yaxis_title="Teaching score",
)
iplot(fig)


### CL14.5: Create a bubble chart
Plot teaching versus rank for 2016, size bubbles by student count and color by international score.

*Hint: You will have to clean the "top25_2016" DataFrame. Look at the "num_students" column; is there an extra string character? Do we need to change datatypes via `astype()`. Can NaN values be plot or do we have to drop them using `dropna()`?*

In [25]:
configure_plotly_browser_state()

bubble_df = top25_2016.copy()
bubble_df["num_students"] = pd.to_numeric(
    bubble_df["num_students"].astype("string").str.replace(",", "", regex=False),
    errors="coerce",
)
for column in ["international", "teaching", "world_rank"]:
    bubble_df[column] = pd.to_numeric(bubble_df[column], errors="coerce")

bubble_df = bubble_df.dropna(
    subset=["num_students", "international", "teaching", "world_rank"]
)
bubble_df = bubble_df.loc[bubble_df["num_students"].gt(0)].copy()
excluded = len(top25_2016) - len(bubble_df)
print(f"Plotting {len(bubble_df)} of the top 25 universities; {excluded} excluded for missing or nonpositive required values.")

bubble_chart = px.scatter(
    bubble_df,
    x="world_rank",
    y="teaching",
    size="num_students",
    color="international",
    size_max=60,
    color_continuous_scale="Viridis",
    hover_name="university_name",
    hover_data={"num_students": ":,.0f", "international": ":.1f"},
    labels={
        "world_rank": "World rank",
        "teaching": "Teaching score",
        "num_students": "Number of students",
        "international": "International score",
    },
    title="Teaching vs Rank: Top 25 Universities in 2016 with Complete Data",
)
bubble_chart.show()


Plotting 24 of the top 25 universities; 1 excluded for missing or nonpositive required values.


**CL14.5 cleaning:** Commas are thousands separators, so `"19,919"`
becomes `19919`, not `19.919`. Numeric conversion uses `errors="coerce"`
to mark invalid entries as missing. Rows missing a required plotting
value are excluded instead of filling them with misleading zeros.
The London School of Economics and Political Science has no student
count in this dataset, leaving 24 bubbles. Bubble area represents
student count, and color represents the international score.
